# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from getpass import getpass

con = duckdb.connect()

# Token entered securely, not pasted in the cell — this repo is public
hf_token = getpass("Paste your Hugging Face READ token: ")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

# Point directly at the month=2026-03 partition rather than scanning the full 79M-row table
month_path = f"{base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

Paste your Hugging Face READ token: ··········


In [2]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, on one report date, for one client — a page-day, from fact_content_daily_performance. I'm using the month=2026-03 partition specifically because it's mid-panel: early months are likely missing GA4 history for many clients (before their ga4_data_start), and the most recent months overlap fact_content_query_90d's fixed 90-day window, which risks leakage if features and label windows aren't aligned — although this notebook doesn't actually use that table so this is just a precaution. A mid-panel month avoids both edge problems.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

#### Feature
gsc_avg_position — knowable at the decision moment because it's the page's measured search position on that specific report_date, not a future or comparative value.

gsc_impressions — knowable because it's a same-day, already-happened measurement.

gsc_clicks — same reasoning as impressions; a direct daily count, no future dependency.

word_count — knowable because it's a static content attribute; less likely than a date field to have been silently overwritten to a post-report-date value, though still worth spot-checking against content_created_date for the same snapshot-vs-history issue.

ga4_sessions (gated on ga4_data_available IS TRUE) — knowable because it reflects site activity that already occurred on that date; used over ga4_engaged_sessions since that column returned zero for 93.6% of GA4-available rows.

#### Label/proxy
is_declining_proxy, built by splitting 2026-03 into two halves (Mar 1-15 vs. Mar 16-31) and comparing gsc_impressions between them. Unlike the starter dataset's trend_direction, there's no pre-built trailing-30-vs-prior-30 comparison in the warehouse, so this is a within-month approximation of the same idea, not a direct equivalent.

Why: the thing being predicted — never also a feature.

#### Context
content_hash_id, client_hash_id, report_date

Why: joining/grouping/splitting only, never model inputs.

#### Excluded
impr_first_half / impr_second_half (self-built, not raw warehouse columns) — the two halves the proxy label is directly computed from. Confirmed by the trap section to leak almost the entire label: adding impr_second_half as a feature pushed ROC AUC from 0.564 to 0.991.

content_updated_date (and anything derived from it, like a "days since last update" feature) — dim_content is a current-state snapshot, not point-in-time history; joining it to daily fact rows produced impossible negative day-counts (updates dated after the report date they were joined to).

Rows where gsc_data_available or ga4_data_available are not explicitly TRUE — confirmed 100% zero/NaN on their respective metrics, so including them without the flag silently mixes "not measured" with "measured zero."

gsc_sum_position — redundant with gsc_avg_position, and a raw sum is hard to interpret correctly without also knowing the row's underlying query count.

client_has_gsc / client_has_ga4 — likely redundant with the row-level gsc_data_available/ga4_data_available flags already used for filtering; excluded as features until it's confirmed whether client-level and row-level availability ever actually diverge.

The ai_chatgpt / ai_perplexity / ai_gemini / ai_copilot / ai_claude / ai_meta / ai_other breakdown columns, and the sessions_organic / sessions_direct / sessions_referral / sessions_social / sessions_paid / sessions_ai breakdown columns — all same-day, decision-moment-safe columns. Not used simply because the assignment caps features at five, not because of any problem found with them.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Grain Check
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM read_parquet('{month_path}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be empty):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []


In [4]:
# Row Count & Date Span

counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{month_path}')
""").df()
print(counts)

    n_rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31


In [5]:
# Availability

availability = con.sql(f"""
    SELECT
      COUNT(*) AS total_rows,
      COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
      COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS excluded_rows
    FROM read_parquet('{month_path}')
""").df()
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  excluded_rows
0     9841378              413966        9427412


In [13]:
# Feature Frame

dim_content_path = f"{base}/dim_content.parquet"
dim_clients_path = f"{base}/dim_clients.parquet"

features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_avg_position,
        f.gsc_impressions,
        f.gsc_clicks,
        d.word_count,
        f.ga4_sessions
    FROM read_parquet('{month_path}') f
    LEFT JOIN read_parquet('{dim_content_path}') d
        ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
      AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS NOT TRUE
""").df()

print(features.shape)
features.head()

(364273, 8)


,client_hash_id,content_hash_id,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,word_count,ga4_sessions
0,client_9958f0a7ae1df715,content_810cf06597918291,2026-03-01,11.272727,11,0,2946,1
1,client_9958f0a7ae1df715,content_eb0aeedbcfaf2712,2026-03-01,5.367347,49,0,3030,1
2,client_9958f0a7ae1df715,content_b813c73d7000b3b1,2026-03-01,5.642857,14,0,3004,1
3,client_9958f0a7ae1df715,content_651b8ba180f9beff,2026-03-01,1.600000,5,0,2978,1
4,client_9958f0a7ae1df715,content_1f39e904c7351258,2026-03-01,7.214286,28,0,2905,1


In [12]:
con.sql(f"""
    SELECT
        AVG(CASE WHEN ga4_engaged_sessions = 0 THEN 1.0 ELSE 0 END) AS pct_zero_engaged,
        AVG(CASE WHEN ga4_sessions = 0 THEN 1.0 ELSE 0 END) AS pct_zero_sessions,
        AVG(ga4_sessions) AS avg_sessions,
        AVG(ga4_engaged_sessions) AS avg_engaged
    FROM read_parquet('{month_path}')
    WHERE ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pct_zero_engaged,pct_zero_sessions,avg_sessions,avg_engaged
0,0.935748,0.008771,3.139891,0.071385


In [24]:
label_df = con.sql(f"""
    WITH halves AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
            SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        content_hash_id,
        impr_first_half,
        impr_second_half,
        CASE WHEN impr_second_half < impr_first_half THEN 1 ELSE 0 END AS is_declining_proxy
    FROM halves
    WHERE impr_first_half > 0
""").df()

print(f"Rows: {len(label_df)}")
print(f"Declining rate: {label_df['is_declining_proxy'].mean():.3f}")

Rows: 151981
Declining rate: 0.438


In [30]:
page_features = con.sql(f"""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS total_impressions,
        SUM(f.gsc_clicks) AS total_clicks,
        AVG(f.ga4_sessions) AS avg_sessions,
        ANY_VALUE(d.word_count) AS word_count
    FROM read_parquet('{month_path}') f
    LEFT JOIN read_parquet('{dim_content_path}') d
        ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND f.ga4_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS NOT TRUE
    GROUP BY f.content_hash_id
""").df()

data = page_features.merge(label_df, on="content_hash_id").fillna(0)
data["total_impressions"] = data["impr_first_half"] + data["impr_second_half"]
print(f"Merged shape: {data.shape}")

Merged shape: (59003, 9)


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = data[["avg_position", "total_impressions", "total_clicks", "avg_sessions", "word_count"]]
y = data["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

honest_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
honest_pipe.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, honest_pipe.predict_proba(X_test)[:, 1])
print(f"Honest ROC AUC (scaled): {honest_score:.3f}")

Honest ROC AUC (scaled): 0.564


In [32]:
data["leaky_second_half_impressions"] = (
    label_df.set_index("content_hash_id")
    .loc[data["content_hash_id"], "impr_second_half"]
    .values
)

X_leaky = data[["avg_position", "total_impressions", "total_clicks",
                 "avg_sessions", "word_count", "leaky_second_half_impressions"]]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.3, random_state=42, stratify=y
)

leaky_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
leaky_pipe.fit(X_train_l, y_train_l)
leaky_score = roc_auc_score(y_test_l, leaky_pipe.predict_proba(X_test_l)[:, 1])
print(f"Leaky ROC AUC (scaled): {leaky_score:.3f}")

Leaky ROC AUC (scaled): 0.991


In [33]:
data = data.drop(columns=["leaky_second_half_impressions"])

print(f"\nLeaky score:  {leaky_score:.3f}  (inflated — built from a column derived from the label itself)")
print(f"Honest score: {honest_score:.3f}  (the real number, using only decision-moment-safe features)")


Leaky score:  0.991  (inflated — built from a column derived from the label itself)
Honest score: 0.564  (the real number, using only decision-moment-safe features)


Adding one label-derived column — impr_second_half, the same second-half impressions figure the label is computed from — pushed ROC AUC from 0.564 to 0.991. This isn't real model improvement; just the model reconstructing the label algebraically, since total_impressions ≈ impr_first_half + impr_second_half and the label is just a comparison of those two halves. Once that column is removed, the honest score of 0.564 is what's left — barely above chance, meaning these five features alone don't yet predict decline well within a single month.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell me anything causal — only association within one month. It can't generalize evenly across clients, since history depth differs wildly (some clients have 12 months, some 3) — a mid-panel month still over-represents clients with longer histories. Rows before a client's ga4_data_start are zero-filled with ga4_data_available = FALSE, so those zeros mean "not measured," not "no engagement" — and a further subset carries a NULL flag, which is neither zero-filled nor explicitly FALSE, so it's excluded outright rather than guessed at. Finally, this month sits mid-panel specifically to avoid fact_content_query_90d's window overlap — but that means this contract doesn't yet prove the label/feature windows stay aligned near the snapshot's edge, which would need a separate check before using a more recent month.

Additionally, dim_content is a current-state snapshot, not a historical record — fields like content_updated_date reflect the latest known value as of export, not the value as of any given report_date. Joining it onto daily fact rows without accounting for this produced negative days_since_last_update values (content "updated" after the report date it was joined to), which is future information leaking backward into what should be a same-day feature. This isn't fixable with a smarter query; it's a genuine limit of the data — any feature built from dim_content needs to be one that's plausible as a fixed, non-time-varying attribute (like word_count), not one that implicitly assumes point-in-time history the table doesn't actually carry.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.